# Manga OCR Dataset Prep on Colab

Ce notebook prépare un workspace dataset complet depuis l'API Mangahere/Consumet jusqu'aux artefacts `page_assets`, `teacher_predictions`, `silver_manifest` et au zip final téléchargeable.


## 1. Préparer le repo et les dépendances

Si le repo est déjà dans Drive, remplace simplement `REPO_DIR` ci-dessous et saute la partie `git clone`.


In [ ]:
REPO_URL = "https://github.com/Ricky-237/manga-ocr.git"  # ex: https://github.com/ton-user/manga-ocr.git
REPO_DIR = "/content/manga-ocr"

from pathlib import Path
import os

if REPO_URL:
    if not Path(REPO_DIR).exists():
        !git clone "$REPO_URL" "$REPO_DIR"
else:
    print("Renseigne REPO_URL ou pointe REPO_DIR vers une copie déjà présente du repo.")

%cd $REPO_DIR
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-eng tesseract-ocr-jpn tesseract-ocr-kor
!python -m pip install -q --upgrade pip
!python -m pip install -q -e .[vision] onnxruntime


## 2. Monter Google Drive si besoin


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 3. Configurer la source manga et le modèle YOLO

Renseigne soit `MANGA_ID`, soit `LATEST_PAGE`. Le chemin `DETECTOR_MODEL_PATH` doit pointer vers ton `yolo26n.onnx` sur Drive ou dans `/content`.


In [ ]:
WORKSPACE_ROOT = "/content/drive/MyDrive/manga-ocr-workdir"
DETECTOR_MODEL_PATH = "/content/drive/MyDrive/models/yolo26n.onnx"

MANGA_ID = None #"one_piece"
LATEST_PAGE = 2
CHAPTER_LIMIT = 8
ALL_CHAPTERS = False

REVIEW_STRATEGY = "exception-audit"
REVIEW_PAGE_LIMIT = 100


## 4. Lancer la préparation du dataset


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "scripts/prepare_colab_dataset.py",
    "--workspace-root", WORKSPACE_ROOT,
    "--detector-model-path", DETECTOR_MODEL_PATH,
    "--review-strategy", REVIEW_STRATEGY,
    "--review-page-limit", str(REVIEW_PAGE_LIMIT),
]

if MANGA_ID:
    cmd.extend(["--manga-id", MANGA_ID, "--chapter-limit", str(CHAPTER_LIMIT)])
    if ALL_CHAPTERS:
        cmd.append("--all-chapters")
elif LATEST_PAGE is not None:
    cmd.extend(["--latest-page", str(LATEST_PAGE)])
else:
    raise ValueError("Renseigne MANGA_ID ou LATEST_PAGE.")

subprocess.run(cmd, check=True)


## 5. Inspecter les artefacts produits


In [ ]:
import json
from pathlib import Path

summary_path = Path(WORKSPACE_ROOT) / "artifacts" / "dataset_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
summary


## 6. Télécharger le zip final


In [ ]:
from google.colab import files
files.download(summary["artifacts"]["archive_path"])
